In [13]:
# Model takes a list of sentences and outputs an array of score with a formality score fore each sentence.




In [14]:
# Imports
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# The word2vec imports

import gensim
import gensim.downloader as api
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

# The LSTM imports
import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn import TransformerEncoder, TransformerEncoderLayer

In [15]:



    

def sentences_to_vectors(sentences, model):
    encoding_dim = model.vector_size
    no_sentence = len(sentences)
    max_word_count = 50
    
    # Initialize a 3D array with zeros
    returned_array = np.zeros((encoding_dim, max_word_count, no_sentence))

    def tokenize(sentence):
        tokens = word_tokenize(sentence)  # Tokenize sentence
        return [word for word in tokens if word not in stopwords.words('english')]  # Remove stopwords

    for i, sentence in enumerate(sentences):
        words = tokenize(sentence)
        word_vectors = [model[word] for word in words if word in model]
        
        # Pad or truncate word_vectors to fit max_word_count
        if len(word_vectors) < max_word_count:
            # Pad with zeros if there are fewer than max_word_count word vectors
            padded_vectors = np.array(word_vectors + [[0] * encoding_dim] * (max_word_count - len(word_vectors)))
        else:
            # Truncate if there are more than max_word_count word vectors
            padded_vectors = np.array(word_vectors[:max_word_count])
        
        # Fill the 3D array
        returned_array[:, :, i] = padded_vectors.T  # Transpose to match shape (encoding_dim, max_word_count)

    return returned_array


def convert_column_sentences_to_vectors(df, column_name, model):
    encoding_dim = model.vector_size
    no_sentence = len(df[column_name])
    max_word_count = 100
    
    # Initialize a 3D array with zeros
    returned_array = np.zeros((encoding_dim, max_word_count, no_sentence))

    meaningless_words = {}  

    def tokenize(sentence):
        tokens = word_tokenize(sentence)  # Tokenize sentence
        return [word for word in tokens if word.lower() not in stopwords.words('english') and word.lower() not in meaningless_words]  # Remove stopwords and meaningless words

    for i, sentence in enumerate(df[column_name]):
        words = tokenize(sentence)
        word_vectors = [model[word] for word in words if word in model]
        
        # Pad or truncate word_vectors to fit max_word_count
        if len(word_vectors) < max_word_count:
            # Pad with zeros if there are fewer than max_word_count word vectors
            padded_vectors = np.array(word_vectors + [[0] * encoding_dim] * (max_word_count - len(word_vectors)))
        else:
            # Truncate if there are more than max_word_count word vectors
            padded_vectors = np.array(word_vectors[:max_word_count])
        
        # Fill the 3D array
        returned_array[:, :, i] = padded_vectors.T  # Transpose to match shape (encoding_dim, max_word_count)

    return returned_array

In [16]:
print(list(api.info()['models'].keys()))

['fasttext-wiki-news-subwords-300', 'conceptnet-numberbatch-17-06-300', 'word2vec-ruscorpora-300', 'word2vec-google-news-300', 'glove-wiki-gigaword-50', 'glove-wiki-gigaword-100', 'glove-wiki-gigaword-200', 'glove-wiki-gigaword-300', 'glove-twitter-25', 'glove-twitter-50', 'glove-twitter-100', 'glove-twitter-200', '__testing_word2vec-matrix-synopsis']


In [26]:
model = api.load('glove-twitter-25')

In [65]:
# Here we load the ready data.

data_dir = r"C:\Users\timur\Documents\GitHub\EmailSentin\data\ready_data.csv"
df_processed_data = pd.read_csv(data_dir)

mapping = {
    'Very Informal': -2,
    'Informal': -1,
    'Neutral': 0,
    'Formal': 1,
    'Very Formal': 2
}


# Encoding the sentences and extracting ground truth array. 

encoded_matrix = convert_column_sentences_to_vectors(df_processed_data, 'text', model)
# Convert the label column to a NumPy array
df_processed_data['numerical_label'] = df_processed_data['label'].map(mapping)
ground_truth = df_processed_data['numerical_label']

# The train test split.

encoded_matrix_transposed = np.transpose(encoded_matrix, (2, 0, 1))  # Rearrange to shape (1333, 25, 25)

# Perform a train-test split
X_train, X_test, y_train, y_test = train_test_split(
    encoded_matrix_transposed, 
    ground_truth, 
    test_size=0.2,  # 20% for testing
    random_state=42,  # For reproducibility
    shuffle=True  # Shuffle the samples before splitting
)

y_train = np.array(y_train)
y_train = np.nan_to_num(y_train, nan=0.0)  # Replace NaNs with 0.0
y_test = np.array(y_test)

X_train = torch.tensor(X_train, dtype=torch.float32)  # Input data (1066 sequences)
y_train = torch.tensor(y_train, dtype=torch.float32)    # Corresponding targets for regression
y_train_binary = np.where(y_train > 0, 1, 0)  # 1 if y_train > 0, otherwise 0


KeyboardInterrupt: 

In [64]:
class BiLSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers):
        super(BiLSTMModel, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        # LSTM layer with bidirectional set to True
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, bidirectional=True)
        
        # Fully connected layer with output size 1 for binary classification
        self.fc = nn.Linear(hidden_size * 2, 1)  # Hidden size * 2 for bidirectional
        
        # Sigmoid activation for binary classification
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        # Initialize hidden and cell states
        h0 = torch.zeros(self.num_layers * 2, x.size(0), self.hidden_size).to(x.device)  # Hidden state
        c0 = torch.zeros(self.num_layers * 2, x.size(0), self.hidden_size).to(x.device)  # Cell state
        
        # LSTM output
        out, _ = self.lstm(x, (h0, c0))
        
        # Get the output of the last time step
        out = out[:, -1, :]  # Shape: (batch_size, hidden_size * 2)
        
        # Pass through the fully connected layer
        out = self.fc(out)  # Shape: (batch_size, 1)
        
        # Apply sigmoid activation to get probabilities
        out = self.sigmoid(out)  # Shape: (batch_size, 1)
        
        return out
    


In [71]:


# Set parameters
input_size = 25    # Number of features
hidden_size = 50   # Number of hidden units in LSTM
num_layers = 1     # Number of LSTM layer
batch_size = 32    # Batch size
num_epochs = 10    # Number of epochs
learning_rate = 0.01  # Learning rate



data_dir = r"C:\Users\timur\Documents\GitHub\EmailSentin\data\ready_data.csv"
df_processed_data = pd.read_csv(data_dir)

mapping = {
    'Very Informal': -2,
    'Informal': -1,
    'Neutral': 0,
    'Formal': 1,
    'Very Formal': 2
}


# Encoding the sentences and extracting ground truth array. 

encoded_matrix = convert_column_sentences_to_vectors(df_processed_data, 'text', model)
# Convert the label column to a NumPy array
df_processed_data['numerical_label'] = df_processed_data['label'].map(mapping)
ground_truth = df_processed_data['numerical_label']

# The train test split.

encoded_matrix_transposed = np.transpose(encoded_matrix, (2, 0, 1))  # Rearrange to shape (1333, 25, 25)

# Perform a train-test split
X_train, X_test, y_train, y_test = train_test_split(
    encoded_matrix_transposed, 
    ground_truth, 
    test_size=0.2,  # 20% for testing
    random_state=42,  # For reproducibility
    shuffle=True  # Shuffle the samples before splitting
)

y_train = np.array(y_train)
y_train = np.nan_to_num(y_train, nan=0.0)  # Replace NaNs with 0.0
y_test = np.array(y_test)
y_train_binary = np.where(y_train > 0, 1, 0)
X_train = torch.tensor(X_train, dtype=torch.float32).transpose(-1, -2)  # Input data (1066 sequences)
y_train = torch.tensor(y_train_binary, dtype=torch.float32)    # Corresponding targets for regression
# 1 if y_train > 0, otherwise 0
# Corresponding targets for regression

# Initialize the model, loss function, and optimizer
modelLSTM = BiLSTMModel(input_size, hidden_size, num_layers)
criterion = nn.MSELoss()  # Mean Squared Error loss for regression
optimizer = optim.Adam(modelLSTM.parameters(), lr=learning_rate)

# Training loop
for epoch in range(num_epochs):
    for i in range(0, X_train.size(0), batch_size):
        # Get the current batch
        batch_data = X_train[i:i + batch_size] 
        
        batch_targets = y_train[i:i + batch_size]  # Shape: (batch_size, 1)

        # Forward pass
        
        outputs = modelLSTM(batch_data)
        
        
        loss = criterion(outputs, batch_targets)
        
        # Backward pass and optimization
        optimizer.zero_grad()  # Zero gradients
        loss.backward()        # Backpropagation
        optimizer.step()       # Update weights

    # Print epoch loss
    print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {loss.item():.4f}')

c:\Users\timur\Documents\GitHub\EmailSentin\env\Lib\site-packages\torch\nn\modules\loss.py:538: UserWarning: Using a target size (torch.Size([32])) that is different to the input size (torch.Size([32, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
c:\Users\timur\Documents\GitHub\EmailSentin\env\Lib\site-packages\torch\nn\modules\loss.py:538: UserWarning: Using a target size (torch.Size([10])) that is different to the input size (torch.Size([10, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Epoch [1/10], Loss: 0.2110
Epoch [2/10], Loss: 0.2100
Epoch [3/10], Loss: 0.2101
Epoch [4/10], Loss: 0.2101
Epoch [5/10], Loss: 0.2100
Epoch [6/10], Loss: 0.2100
Epoch [7/10], Loss: 0.2100
Epoch [8/10], Loss: 0.2101
Epoch [9/10], Loss: 0.2101
Epoch [10/10], Loss: 0.2101


In [67]:
batch_targets

array([0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 1, 0,
       0, 0, 1, 0, 0, 0, 0, 0, 0, 0])